# Raw API Data Ingestion — Environmental Data Lakehouse

This notebook performs **raw ingestion** of environmental data from three APIs:
1. **Open-Meteo Weather API** — comprehensive current, hourly, and daily forecast variables
2. **Open-Meteo Air Quality API** — pollutants, pollen, UV, and AQI indices
3. **USGS Earthquake API** — complete GeoJSON feed (all_day summary)

Raw JSON responses are saved to `/Volumes/weather_api/raw/raw_files/` in source-specific subdirectories.
Each extraction produces a uniquely named, timestamped file. No transformations are applied.

**Naming convention:**
- `weather_<Location>_<YYYYMMDDTHHMMSSZ>.json`
- `air_quality_<Location>_<YYYYMMDDTHHMMSSZ>.json`
- `earthquakes_<YYYYMMDDTHHMMSSZ>.json`

In [0]:
import json
import os
import re
import time
from datetime import datetime, timezone
from pathlib import Path

import requests
import yaml
from dotenv import load_dotenv

# ── Project paths ──
PROJECT_ROOT = Path("/Workspace/Users/adityarajsharma3030@gmail.com/weather-data-pipeline")
LOCATIONS_FILE = PROJECT_ROOT / "config" / "locations.yaml"

# Load API URLs from .env
load_dotenv(PROJECT_ROOT / ".env")
WEATHER_API_URL = os.getenv("WEATHER_API_URL")
AIR_QUALITY_API_URL = os.getenv("AIR_QUALITY_API_URL")
USGS_API_URL = os.getenv("USGS_API_URL")

# ── Storage paths ──
VOLUME_ROOT = Path("/Volumes/weather_api/raw/raw_files")
WEATHER_DIR = VOLUME_ROOT / "weather"
AIR_QUALITY_DIR = VOLUME_ROOT / "air_quality"
EARTHQUAKES_DIR = VOLUME_ROOT / "earthquakes"

# ── Weather API variables (verified against Open-Meteo docs) ──
WEATHER_CURRENT_VARS = ",".join([
    "temperature_2m", "relative_humidity_2m", "apparent_temperature",
    "is_day", "precipitation", "rain", "snowfall", "weather_code",
    "cloud_cover", "pressure_msl", "surface_pressure",
    "wind_speed_10m", "wind_direction_10m", "wind_gusts_10m",
])

WEATHER_HOURLY_VARS = ",".join([
    "temperature_2m", "relative_humidity_2m", "dew_point_2m",
    "apparent_temperature", "precipitation", "rain", "snowfall",
    "snow_depth", "weather_code", "pressure_msl", "surface_pressure",
    "cloud_cover", "cloud_cover_low", "cloud_cover_mid", "cloud_cover_high",
    "visibility", "wind_speed_10m", "wind_speed_80m",
    "wind_direction_10m", "wind_direction_80m", "wind_gusts_10m",
    "soil_temperature_0cm", "soil_temperature_6cm", "soil_temperature_18cm", "soil_temperature_54cm",
    "soil_moisture_0_to_1cm", "soil_moisture_1_to_3cm", "soil_moisture_3_to_9cm",
    "soil_moisture_9_to_27cm", "soil_moisture_27_to_81cm",
    "uv_index", "uv_index_clear_sky", "is_day", "sunshine_duration",
    "wet_bulb_temperature_2m", "vapour_pressure_deficit",
    "shortwave_radiation", "direct_radiation", "diffuse_radiation",
    "direct_normal_irradiance", "terrestrial_radiation",
])

WEATHER_DAILY_VARS = ",".join([
    "weather_code", "temperature_2m_max", "temperature_2m_min",
    "apparent_temperature_max", "apparent_temperature_min",
    "sunrise", "sunset", "daylight_duration", "sunshine_duration",
    "uv_index_max", "precipitation_sum", "rain_sum", "snowfall_sum",
    "precipitation_hours", "precipitation_probability_max",
    "wind_speed_10m_max", "wind_gusts_10m_max", "wind_direction_10m_dominant",
    "shortwave_radiation_sum", "et0_fao_evapotranspiration",
])

# ── Air Quality API variables (verified — European AQI sub-indices NOT supported) ──
AQ_CURRENT_VARS = ",".join([
    "pm10", "pm2_5", "carbon_monoxide", "nitrogen_dioxide",
    "sulphur_dioxide", "ozone", "aerosol_optical_depth", "dust",
    "uv_index", "uv_index_clear_sky", "ammonia",
    "alder_pollen", "birch_pollen", "grass_pollen",
    "mugwort_pollen", "olive_pollen", "ragweed_pollen",
    "european_aqi", "us_aqi",
    "us_aqi_pm2_5", "us_aqi_pm10", "us_aqi_ozone",
    "us_aqi_nitrogen_dioxide", "us_aqi_sulphur_dioxide", "us_aqi_carbon_monoxide",
])

AQ_HOURLY_VARS = AQ_CURRENT_VARS  # Same variables available hourly

print("✅ Configuration loaded")
print(f"   Weather API:     {WEATHER_API_URL}")
print(f"   Air Quality API: {AIR_QUALITY_API_URL}")
print(f"   USGS API:        {USGS_API_URL}")
print(f"   Storage root:    {VOLUME_ROOT}")

In [0]:
def load_locations():
    """Load and deduplicate locations from config/locations.yaml."""
    with open(LOCATIONS_FILE) as f:
        config = yaml.safe_load(f)
    locations = config["locations"]
    # Deduplicate by (name, latitude, longitude)
    seen = set()
    unique = []
    for loc in locations:
        key = (loc["name"], loc["latitude"], loc["longitude"])
        if key not in seen:
            seen.add(key)
            unique.append(loc)
    if len(unique) < len(locations):
        print(f"⚠️  Removed {len(locations) - len(unique)} duplicate location(s) from config")
    return unique


def sanitize_location_name(name):
    """Convert a location name to a filename-safe string."""
    sanitized = re.sub(r"[^\w\s-]", "", name)
    return re.sub(r"[\s-]+", "_", sanitized).strip("_")


def get_timestamp_utc():
    """Return current UTC timestamp in ISO and filename formats."""
    now = datetime.now(timezone.utc)
    return now.isoformat(), now.strftime("%Y%m%dT%H%M%SZ")


def save_raw_json(payload, filepath):
    """Write JSON atomically — write to temp file, validate, then rename."""
    temp_path = str(filepath) + ".tmp"
    try:
        with open(temp_path, "w", encoding="utf-8") as f:
            json.dump(payload, f, indent=2, ensure_ascii=False)
        # Validate the temp file is parseable JSON before committing
        with open(temp_path, "r", encoding="utf-8") as f:
            json.load(f)
        os.rename(temp_path, str(filepath))
    except Exception:
        if os.path.exists(temp_path):
            os.remove(temp_path)
        raise


def build_wrapper(source, raw_response, api_endpoint, location_name=None,
                   latitude=None, longitude=None, request_params=None):
    """Wrap the raw API response with ingestion metadata (does not modify the payload)."""
    iso_ts, _ = get_timestamp_utc()
    wrapper = {
        "metadata": {
            "source": source,
            "retrieval_timestamp_utc": iso_ts,
            "api_endpoint": api_endpoint,
        },
        "raw_response": raw_response,
    }
    if location_name is not None:
        wrapper["metadata"]["location_name"] = location_name
    if latitude is not None:
        wrapper["metadata"]["latitude"] = latitude
    if longitude is not None:
        wrapper["metadata"]["longitude"] = longitude
    if request_params is not None:
        wrapper["metadata"]["request_params"] = request_params
    return wrapper

In [0]:
def ingest_weather(locations):
    """Fetch comprehensive weather data for each location and save raw JSON."""
    print(f"\n{'='*60}")
    print(f"  WEATHER INGESTION — {len(locations)} locations")
    print(f"{'='*60}")

    results = {"success": [], "errors": []}

    for loc in locations:
        loc_name = loc["name"]
        try:
            params = {
                "latitude": loc["latitude"],
                "longitude": loc["longitude"],
                "current": WEATHER_CURRENT_VARS,
                "hourly": WEATHER_HOURLY_VARS,
                "daily": WEATHER_DAILY_VARS,
                "forecast_days": 1,
                "timezone": "UTC",
            }

            response = requests.get(WEATHER_API_URL, params=params, timeout=30)
            response.raise_for_status()
            raw_data = response.json()

            # Validate response has expected structure
            if not raw_data or "latitude" not in raw_data:
                raise ValueError("Empty or malformed weather response")

            wrapper = build_wrapper(
                source="weather",
                raw_response=raw_data,
                api_endpoint=WEATHER_API_URL,
                location_name=loc_name,
                latitude=loc["latitude"],
                longitude=loc["longitude"],
                request_params=params,
            )

            _, file_ts = get_timestamp_utc()
            safe_name = sanitize_location_name(loc_name)
            filename = f"weather_{safe_name}_{file_ts}.json"
            filepath = WEATHER_DIR / filename
            save_raw_json(wrapper, filepath)

            print(f"✅ weather — {loc_name}: {filepath}")
            results["success"].append({"location": loc_name, "path": str(filepath)})

        except Exception as e:
            print(f"❌ weather — {loc_name}: {e}")
            results["errors"].append({"location": loc_name, "error": str(e)})

    return results

In [0]:
def ingest_air_quality(locations):
    """Fetch comprehensive air quality data for each location and save raw JSON."""
    print(f"\n{'='*60}")
    print(f"  AIR QUALITY INGESTION — {len(locations)} locations")
    print(f"{'='*60}")

    results = {"success": [], "errors": []}

    for loc in locations:
        loc_name = loc["name"]
        try:
            params = {
                "latitude": loc["latitude"],
                "longitude": loc["longitude"],
                "current": AQ_CURRENT_VARS,
                "hourly": AQ_HOURLY_VARS,
                "timezone": "UTC",
                "forecast_days": 1,
            }

            response = requests.get(AIR_QUALITY_API_URL, params=params, timeout=30)
            response.raise_for_status()
            raw_data = response.json()

            # Validate response has expected structure
            if not raw_data or "latitude" not in raw_data:
                raise ValueError("Empty or malformed air quality response")

            wrapper = build_wrapper(
                source="air_quality",
                raw_response=raw_data,
                api_endpoint=AIR_QUALITY_API_URL,
                location_name=loc_name,
                latitude=loc["latitude"],
                longitude=loc["longitude"],
                request_params=params,
            )

            _, file_ts = get_timestamp_utc()
            safe_name = sanitize_location_name(loc_name)
            filename = f"air_quality_{safe_name}_{file_ts}.json"
            filepath = AIR_QUALITY_DIR / filename
            save_raw_json(wrapper, filepath)

            print(f"✅ air_quality — {loc_name}: {filepath}")
            results["success"].append({"location": loc_name, "path": str(filepath)})

        except Exception as e:
            print(f"❌ air_quality — {loc_name}: {e}")
            results["errors"].append({"location": loc_name, "error": str(e)})

    return results

In [0]:
def ingest_earthquakes():
    """Fetch the complete USGS earthquake GeoJSON feed and save raw JSON."""
    print(f"\n{'='*60}")
    print(f"  EARTHQUAKE INGESTION — USGS all_day feed")
    print(f"{'='*60}")

    results = {"success": [], "errors": []}

    try:
        response = requests.get(USGS_API_URL, timeout=30)
        response.raise_for_status()
        raw_data = response.json()

        # Validate response has GeoJSON structure
        if not raw_data or "features" not in raw_data:
            raise ValueError("Empty or malformed earthquake GeoJSON response")

        feature_count = len(raw_data.get("features", []))

        # Wrap with metadata (no location — global feed)
        wrapper = build_wrapper(
            source="earthquakes",
            raw_response=raw_data,
            api_endpoint=USGS_API_URL,
        )

        _, file_ts = get_timestamp_utc()
        filename = f"earthquakes_{file_ts}.json"
        filepath = EARTHQUAKES_DIR / filename
        save_raw_json(wrapper, filepath)

        print(f"✅ earthquakes — {feature_count} events: {filepath}")
        results["success"].append({"events": feature_count, "path": str(filepath)})

    except Exception as e:
        print(f"❌ earthquakes: {e}")
        results["errors"].append({"error": str(e)})

    return results

In [0]:
def main():
    """Run all three ingestion sources and report results."""
    print("=" * 60)
    print("  RAW API DATA INGESTION — Environmental Data Lakehouse")
    print(f"  Run time (UTC): {datetime.now(timezone.utc).isoformat()}")
    print("=" * 60)

    # Verify storage directories exist
    for d in [WEATHER_DIR, AIR_QUALITY_DIR, EARTHQUAKES_DIR]:
        d.mkdir(parents=True, exist_ok=True)

    # Verify API URLs are loaded
    for name, url in [("WEATHER_API_URL", WEATHER_API_URL),
                      ("AIR_QUALITY_API_URL", AIR_QUALITY_API_URL),
                      ("USGS_API_URL", USGS_API_URL)]:
        if not url:
            raise RuntimeError(f"{name} is not set — check .env at {PROJECT_ROOT / '.env'}")

    # Load and deduplicate locations
    locations = load_locations()
    print(f"\n📡 Loaded {len(locations)} unique locations from config")

    # Run ingestion — each source is independent, failures don't block others
    all_results = {}

    try:
        all_results["weather"] = ingest_weather(locations)
    except Exception as e:
        print(f"❌ Weather ingestion failed entirely: {e}")
        all_results["weather"] = {"success": [], "errors": [{"error": str(e)}]}

    time.sleep(1)  # Brief pause to respect rate limits between sources

    try:
        all_results["air_quality"] = ingest_air_quality(locations)
    except Exception as e:
        print(f"❌ Air quality ingestion failed entirely: {e}")
        all_results["air_quality"] = {"success": [], "errors": [{"error": str(e)}]}

    time.sleep(1)

    try:
        all_results["earthquakes"] = ingest_earthquakes()
    except Exception as e:
        print(f"❌ Earthquake ingestion failed entirely: {e}")
        all_results["earthquakes"] = {"success": [], "errors": [{"error": str(e)}]}

    # Summary
    print(f"\n{'='*60}")
    print("  INGESTION SUMMARY")
    print(f"{'='*60}")
    for source, res in all_results.items():
        s = len(res["success"])
        e = len(res["errors"])
        print(f"  {source:15s} — ✅ {s} succeeded, ❌ {e} failed")

    return all_results


# Run the full ingestion
results = main()

In [0]:
def validate_ingestion(results):
    """Verify saved files are valid JSON and contain complete API responses."""
    print(f"\n{'='*60}")
    print("  VALIDATION")
    print(f"{'='*60}")

    all_files = []
    for source in ["weather", "air_quality", "earthquakes"]:
        source_dir = VOLUME_ROOT / source
        if source_dir.exists():
            files = sorted(source_dir.glob("*.json"))
            all_files.extend(files)
            print(f"\n📁 {source}/ — {len(files)} file(s)")
            for f in files[-3:]:  # Show last 3
                print(f"   {f.name}")

    print(f"\n📋 Validating {len(all_files)} file(s)...")
    valid = 0
    invalid = 0

    for f in all_files:
        try:
            with open(f, "r", encoding="utf-8") as fh:
                data = json.load(fh)

            # Check for expected wrapper structure
            if "metadata" not in data or "raw_response" not in data:
                print(f"❌ {f.name}: missing wrapper structure")
                invalid += 1
                continue

            # Check raw_response is non-empty
            if not data["raw_response"]:
                print(f"❌ {f.name}: empty raw_response")
                invalid += 1
                continue

            valid += 1

        except json.JSONDecodeError:
            print(f"❌ {f.name}: invalid JSON")
            invalid += 1
        except Exception as e:
            print(f"❌ {f.name}: {e}")
            invalid += 1

    print(f"\n  ✅ Valid files: {valid}")
    print(f"  ❌ Invalid files: {invalid}")

    # Show sample file contents (metadata + key counts)
    if all_files:
        sample = all_files[-1]
        with open(sample, "r", encoding="utf-8") as fh:
            sample_data = json.load(fh)
        print(f"\n📋 Sample file: {sample.name}")
        print(f"   Source: {sample_data['metadata']['source']}")
        print(f"   Retrieved: {sample_data['metadata']['retrieval_timestamp_utc']}")
        raw = sample_data["raw_response"]
        if "current" in raw:
            print(f"   Current fields: {len(raw.get('current', {}))}")
        if "hourly" in raw:
            print(f"   Hourly fields: {len(raw.get('hourly', {}))}")
            print(f"   Hourly records: {len(raw.get('hourly', {}).get('time', []))}")
        if "daily" in raw:
            print(f"   Daily fields: {len(raw.get('daily', {}))}")
        if "features" in raw:
            print(f"   Earthquake features: {len(raw.get('features', []))}")

    print(f"\n{'='*60}")
    print("  VALIDATION COMPLETE")
    print(f"{'='*60}")

    return {"valid": valid, "invalid": invalid, "total_files": len(all_files)}


validation = validate_ingestion(results)